# Airplane Distribution Problem in Pure Python
## Structural Exact Reasoning for Book Problem 3.4

This notebook solves the airplane-loading problem from book section `3.4` with pure Python and exact structural arguments.

The key observations are:
1. equal loading proportion across compartments only constrains the compartment weights,
2. even if every compartment is filled at `100 %` of its weight capacity, the volume limits remain slack,
3. once the full `50-ton` capacity is confirmed feasible, the optimal cargo mix is obtained by taking the highest-utility tons first.


In [1]:
from __future__ import annotations

from time import perf_counter


In [2]:
COMPARTMENTS = {
    "front": {"weight": 16, "volume": 200},
    "center": {"weight": 20, "volume": 250},
    "rear": {"weight": 14, "volume": 150},
}
CARGO = {
    "tools": {"available": 20, "volume": 1, "utility": 250},
    "books": {"available": 15, "volume": 2, "utility": 280},
    "flowers": {"available": 8, "volume": 10, "utility": 500},
    "sculptures": {"available": 10, "volume": 6, "utility": 360},
}
EXPECTED_TOTALS = {"tools": 17, "books": 15, "flowers": 8, "sculptures": 10}
EXPECTED_UTILITY = 16_050
EXPECTED_G = 1.0


def elapsed_call(func, *args, **kwargs):
    start = perf_counter()
    result = func(*args, **kwargs)
    elapsed = perf_counter() - start
    return result, elapsed


def utility_of(loads_by_type):
    return sum(CARGO[cargo]["utility"] * amount for cargo, amount in loads_by_type.items())


def verify_layout(layout):
    type_totals = {cargo: 0 for cargo in CARGO}
    for compartment, loads in layout.items():
        weight = sum(loads.values())
        volume = sum(loads[cargo] * CARGO[cargo]["volume"] for cargo in CARGO)
        if weight > COMPARTMENTS[compartment]["weight"] + 1e-9:
            return False
        if volume > COMPARTMENTS[compartment]["volume"] + 1e-9:
            return False
        for cargo, amount in loads.items():
            type_totals[cargo] += amount
    if any(type_totals[cargo] > CARGO[cargo]["available"] for cargo in CARGO):
        return False
    target_ratio = None
    for compartment, loads in layout.items():
        ratio = sum(loads.values()) / COMPARTMENTS[compartment]["weight"]
        if target_ratio is None:
            target_ratio = ratio
        elif abs(ratio - target_ratio) > 1e-9:
            return False
    return True


## 1. Capacity-Ratio Check


In [3]:
volume_per_ton_limit = min(
    COMPARTMENTS[compartment]["volume"] / COMPARTMENTS[compartment]["weight"]
    for compartment in COMPARTMENTS
)
max_cargo_volume = max(CARGO[cargo]["volume"] for cargo in CARGO)

print("Minimum compartment volume-per-ton ratio:", volume_per_ton_limit)
print("Maximum cargo volume-per-ton ratio:", max_cargo_volume)

assert volume_per_ton_limit >= max_cargo_volume
print("Volume never becomes binding before weight in any compartment.")


Minimum compartment volume-per-ton ratio: 10.714285714285714
Maximum cargo volume-per-ton ratio: 10
Volume never becomes binding before weight in any compartment.


## 2. Optimal Cargo Mix


In [4]:
def solve_cargo_mix():
    remaining_capacity = sum(COMPARTMENTS[compartment]["weight"] for compartment in COMPARTMENTS)
    selected = {}
    for cargo, data in sorted(CARGO.items(), key=lambda item: item[1]["utility"], reverse=True):
        take = min(data["available"], remaining_capacity)
        selected[cargo] = take
        remaining_capacity -= take
    return selected


cargo_mix, mix_time = elapsed_call(solve_cargo_mix)
print("Selected cargo totals:", cargo_mix)
print(f"Elapsed time: {mix_time:.8f} seconds")
print("Utility:", utility_of(cargo_mix))

assert cargo_mix == EXPECTED_TOTALS
assert utility_of(cargo_mix) == EXPECTED_UTILITY


Selected cargo totals: {'flowers': 8, 'sculptures': 10, 'books': 15, 'tools': 17}
Elapsed time: 0.00000467 seconds
Utility: 16050


## 3. One Feasible Optimal Layout


In [5]:
def construct_layout():
    return {
        "front": {"tools": 8, "books": 0, "flowers": 8, "sculptures": 0},
        "center": {"tools": 5, "books": 15, "flowers": 0, "sculptures": 0},
        "rear": {"tools": 4, "books": 0, "flowers": 0, "sculptures": 10},
    }


layout, layout_time = elapsed_call(construct_layout)
print("Layout:", layout)
print(f"Elapsed time: {layout_time:.8f} seconds")
assert verify_layout(layout)

compartment_loads = {compartment: sum(layout[compartment].values()) for compartment in layout}
load_ratio = {compartment: compartment_loads[compartment] / COMPARTMENTS[compartment]["weight"] for compartment in layout}
print("Compartment loads:", compartment_loads)
print("Load ratios:", load_ratio)
print("Total utility:", utility_of(cargo_mix))

assert all(abs(load_ratio[compartment] - EXPECTED_G) < 1e-9 for compartment in load_ratio)
print("\nThe full-load plan reproduces the published objective value of 16,050.")


Layout: {'front': {'tools': 8, 'books': 0, 'flowers': 8, 'sculptures': 0}, 'center': {'tools': 5, 'books': 15, 'flowers': 0, 'sculptures': 0}, 'rear': {'tools': 4, 'books': 0, 'flowers': 0, 'sculptures': 10}}
Elapsed time: 0.00000063 seconds
Compartment loads: {'front': 16, 'center': 20, 'rear': 14}
Load ratios: {'front': 1.0, 'center': 1.0, 'rear': 1.0}
Total utility: 16050

The full-load plan reproduces the published objective value of 16,050.


## Note on Multiple Layouts

The transported tonnage by cargo type is unique at optimum, but several compartment-by-compartment allocations can achieve the same utility once all compartments are fully loaded with the same weight proportion. The layout above is one explicit optimal arrangement.
